In [2]:
# Defining project paths
BASE_DIR = "D:/Capstone/capstone_repo"
DATA_DIR = f"{BASE_DIR}/data"
NB_DIR = f"{BASE_DIR}/notebooks"

import os
os.makedirs(DATA_DIR, exist_ok=True)

In [3]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from shapely.geometry import Point
from shapely.ops import nearest_points

# Analysis

In [4]:
healthcare = pd.read_csv(f"{DATA_DIR}/processed/casablanca_healthcare_cleaned.csv")
stops = pd.read_csv(f"{DATA_DIR}/processed/casablanca_transport_stops_cleaned.csv")

In [5]:
# Convert to GeoDataFrames
healthcare_gdf = gpd.GeoDataFrame(
    healthcare,
    geometry=gpd.points_from_xy(healthcare.lon, healthcare.lat),
    crs="EPSG:4326"
)

stops_gdf = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops.lon, stops.lat),
    crs="EPSG:4326"
)


In [6]:
# Project to UTM
healthcare_gdf = healthcare_gdf.to_crs(epsg=32629)
stops_gdf = stops_gdf.to_crs(epsg=32629)

In [7]:
# Compute nearest stop distance
def nearest_distance(point, stops_geom):
    nearest_geom = stops_geom.geometry.unary_union
    nearest_point = nearest_points(point, nearest_geom)[1]
    return point.distance(nearest_point)

healthcare_gdf['distance_to_stop_m'] = healthcare_gdf.geometry.apply(
    lambda x: nearest_distance(x, stops_gdf)
)

C:\Users\afafb\AppData\Local\Temp\ipykernel_9872\1660183459.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  nearest_geom = stops_geom.geometry.unary_union


In [8]:
# Convert meters to minutes
healthcare_gdf['walking_time_min'] = healthcare_gdf['distance_to_stop_m'] / 83.3

In [10]:
# Results
healthcare_gdf[['name', 'distance_to_stop_m', 'walking_time_min']].head()


,name,distance_to_stop_m,walking_time_min
0,Clinique Badr مصحة بدر,312.197016,3.747863
1,clinique dentaire casablanca (cdc),30.472442,0.365816
2,Centre consultation et traitement dentaires,208.219746,2.499637
3,Clinique d'accouchement,149.184204,1.790927
4,Hôpital Sidi Othmane,202.572789,2.431846


In [11]:
healthcare_gdf['accessible_5min'] = healthcare_gdf['walking_time_min'] <= 5
healthcare_gdf['accessible_5min'].value_counts(normalize=True) * 100

accessible_5min
True     99.047619
False     0.952381
Name: proportion, dtype: float64

# Visualization

In [12]:
# Convert your data back to WGS84 (for mapping)
healthcare_map = healthcare_gdf.to_crs(epsg=4326)
stops_map = stops_gdf.to_crs(epsg=4326)

In [14]:
# create an interactive map using folium
import folium

# Center map on Casablanca
m = folium.Map(location=[33.5731, -7.5898], zoom_start=12)

# Add transport stops (blue)
for _, row in stops_map.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color='blue',
        fill=True,
        fill_opacity=0.5,
        popup=row['name']
    ).add_to(m)

# Add healthcare facilities with accessibility colors
for _, row in healthcare_map.iterrows():
    if row['walking_time_min'] <= 5:
        color = 'green'   # good access
    elif row['walking_time_min'] <= 10:
        color = 'orange'  # medium access
    else:
        color = 'red'     # poor access

    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=f"{row['name']}<br>Time: {row['walking_time_min']:.2f} min",
        icon=folium.Icon(color=color)
    ).add_to(m)

m


In [15]:
m.save("casablanca_accessibility_map.html")

# Analysis by district

In [ ]:
districts = gpd.read_file("gadm41_MAR_3.shp")  # example name
districts.head()
